# Experiment V3: YOLOv8 Shallower Backbone for Object Detection

**Student ID:** 25509225  
**Experiment:** V3 - YOLOv8 with Shallower Backbone  
**Model:** YOLOv8m with reduced C2f module repeats

---

## Overview

This experiment trains a YOLOv8m model with a shallower backbone architecture. The C2f module at backbone index 6 has its repeat count halved to create a lighter model with faster inference. This is compared with the baseline (V1) and deeper backbone (V2) models.

## Cell 1: Load Modules

In [ ]:
# Load shared modules from YOLOv8_modules.ipynb
%run ./YOLOv8_modules.ipynb

print("✓ All modules loaded successfully")

## Cell 2: Configuration

In [ ]:
# ============================================================================
# Experiment V3: YOLOv8 Shallower Backbone Configuration
# ============================================================================

# === Model Configuration ===
YOLOV8_V3_MODEL_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'shallower'  # Reduces C2f repeats in backbone
}

# === Training Configuration ===
TRAINING_CONFIG_V3 = {
    'learning_rate': 0.001,        # Standard LR for lighter model
    'batch_size': 16,              # Larger batch possible due to smaller model
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 1e-4,          # Standard weight decay
    'use_amp': True,               # Mixed precision
    'patience': 50,                # Early stopping patience
    'cos_lr': False,               # No cosine LR schedule
    'close_mosaic': 0              # Keep mosaic augmentation throughout
}

# === Experiment Settings ===
STUDENT_ID = "25509225"
DATASET_CONFIG = f"/home/sagemaker-user/CNN_A2/data/{STUDENT_ID}/Object_Detection/yolo/data.yaml"
USE_PRETRAINED = True

# Output directory (simplified, no timestamp subdirectory)
output_dir = Path(f'outputs/detection_yolov8_v3')
output_dir.mkdir(parents=True, exist_ok=True)

# Print experiment info
print("=" * 80)
print("EXPERIMENT V3: YOLOv8 Shallower Backbone")
print("=" * 80)
print(f"\nExperiment Settings:")
print(f"  Student ID: {STUDENT_ID}")
print(f"  Dataset Config: {DATASET_CONFIG}")
print(f"  Output Directory: {output_dir}")
print(f"\nModel Configuration:")
print(f"  Backbone: YOLOv8{YOLOV8_V3_MODEL_CONFIG['backbone']} + Shallower")
print(f"  Input Size: {YOLOV8_V3_MODEL_CONFIG['input_size']}")
print(f"  Pretrained: {USE_PRETRAINED}")
print(f"  Customization: Shallower (halved C2f repeats)")
print(f"\nTraining Configuration:")
print(f"  Learning Rate: {TRAINING_CONFIG_V3['learning_rate']}")
print(f"  Batch Size: {TRAINING_CONFIG_V3['batch_size']}")
print(f"  Epochs: {TRAINING_CONFIG_V3['epochs']}")
print(f"  Optimizer: {TRAINING_CONFIG_V3['optimizer']}")
print(f"  Weight Decay: {TRAINING_CONFIG_V3['weight_decay']}")
print(f"  Mixed Precision: {TRAINING_CONFIG_V3['use_amp']}")
print(f"  Early Stopping: {TRAINING_CONFIG_V3['patience']} epochs")
print(f"  Cosine LR: {TRAINING_CONFIG_V3['cos_lr']}")
print(f"  Close Mosaic: {TRAINING_CONFIG_V3['close_mosaic']} epochs")

## Cell 3: Step 1 - Load Dataset Configuration

In [ ]:
# ============================================================================
# Step 1: Load Dataset Configuration
# ============================================================================

print("\n[1/5] Loading dataset configuration...")
print("=" * 80)

dataset_config_path = Path(DATASET_CONFIG)

if not dataset_config_path.exists():
    print(f"Error: Dataset config not found: {dataset_config_path}")
    print("Please check the DATASET_CONFIG path in Cell 2")
    raise FileNotFoundError(f"Dataset config not found: {dataset_config_path}")

with open(dataset_config_path, 'r') as f:
    dataset_config = yaml.safe_load(f)

print(f"Dataset config: {dataset_config_path}")
print(f"Number of classes: {dataset_config['nc']}")
print(f"Class names: {dataset_config['names']}")
print(f"\nPaths:")
print(f"  Train: {dataset_config.get('train', 'N/A')}")
print(f"  Val: {dataset_config.get('val', 'N/A')}")
print(f"  Test: {dataset_config.get('test', 'N/A')}")
print("\n✓ Dataset configuration loaded successfully")

## Cell 4: Step 2 - Initialize Model

In [ ]:
# ============================================================================
# Step 2: Initialize YOLOv8 Model with Shallower Backbone
# ============================================================================

print("\n[2/5] Initializing YOLOv8 model with shallower backbone...")
print("=" * 80)

# Update model config with the pretrained setting
model_config = YOLOV8_V3_MODEL_CONFIG.copy()
model_config['pretrained'] = USE_PRETRAINED

# Create model
model = YOLOv8Detector(**model_config)

print(f"\nModel Information:")
print(f"  Model: YOLOv8{model_config['backbone']} + Shallower Backbone")
print(f"  Input Size: {model_config['input_size']}")
print(f"  Pretrained: {USE_PRETRAINED}")
print(f"  Customization: Shallower (C2f repeats halved at layer 6)")
print(f"  Confidence Threshold: {model_config['confidence_threshold']}")
print(f"  NMS IoU Threshold: {model_config['nms_iou_threshold']}")

# Calculate approximate parameters
total_params = sum(p.numel() for p in model.model.parameters())
print(f"\n  Total Parameters: ~{total_params/1e6:.1f}M")
print("\n✓ Model initialized successfully")

## Cell 5: Step 3 - Train Model

In [ ]:
# ============================================================================
# Step 3: Train Model
# ============================================================================

print("\n[3/5] Training model...")
print("=" * 80)

# Initialize trainer
trainer = YOLOv8Trainer(**TRAINING_CONFIG_V3)

# Train model
training_results = trainer.train(
    model=model,
    train_data=DATASET_CONFIG,
    val_data=DATASET_CONFIG,
    output_dir=str(output_dir)
)

# Print training summary
print("\n" + "=" * 80)
print("TRAINING COMPLETED")
print("=" * 80)
print(f"\nTraining Results:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")
print(f"  Output Directory: {output_dir}")
print("\n✓ Model training completed successfully")

## Cell 6: Step 4 - Evaluate on Test Set

In [ ]:
# ============================================================================
# Step 4: Evaluate on Test Set
# ============================================================================

print("\n[4/5] Evaluating on test set...")
print("=" * 80)

# Initialize evaluator
evaluator = DetectionEvaluator()

# Evaluate model
metrics = evaluator.evaluate_yolov8(
    model=model,
    test_dataset=DATASET_CONFIG
)

print("\n" + "=" * 80)
print("EVALUATION COMPLETED")
print("=" * 80)
print(f"\nTest Set Metrics:")
print(f"  mAP@0.5: {metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"  F1-Score: {metrics['f1']:.4f}")
print("\n✓ Model evaluation completed successfully")

## Cell 7: Display Detection Results

In [ ]:
# ============================================================================
# Visualization 1: Detection Results with Bounding Boxes
# ============================================================================

print("\n=== Detection Results Visualization ===")
print("=" * 80)

# Plot detection results
fig = evaluator.plot_detection_results(model, DATASET_CONFIG, num_samples=5)
if fig is not None:
    plt.show()
    print("\n✓ Detection visualization displayed")
else:
    print("Warning: Could not generate detection visualization")

## Cell 8: Display Training Curves

In [ ]:
# ============================================================================
# Visualization 2: Training Curves (Loss, mAP, Precision, Recall, F1)
# ============================================================================

print("\n=== Training Curves ===")
print("=" * 80)

# Plot training curves
fig = evaluator.plot_training_curves(training_results)
if fig is not None:
    plt.show()
    print("\n✓ Training curves displayed")
else:
    print("Warning: Could not generate training curves")

## Cell 9: Display PR Curves

In [ ]:
# ============================================================================
# Visualization 3: Precision-Recall Curves
# ============================================================================

print("\n=== Precision-Recall Curves ===")
print("=" * 80)

# Plot PR curves
fig = evaluator.plot_pr_curves(training_results)
if fig is not None:
    plt.show()
    print("\n✓ PR curves displayed")
else:
    print("Warning: Could not generate PR curves")

## Cell 10: Model Analysis

In [ ]:
# ============================================================================
# Analysis: Model Characteristics and Performance
# ============================================================================

print("\n" + "=" * 80)
print("MODEL ANALYSIS")
print("=" * 80)

# Model characteristics
total_params = sum(p.numel() for p in model.model.parameters())

print(f"\nModel Characteristics:")
print(f"  Backbone: YOLOv8{model_config['backbone']} + Shallower")
print(f"  Input Size: {model_config['input_size']}")
print(f"  Total Parameters: ~{total_params/1e6:.1f}M")
print(f"  Customization: Shallower Backbone (C2f repeats halved)")
print(f"  Pretrained Weights: {USE_PRETRAINED}")

print(f"\nPerformance Summary:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")
print(f"  Test mAP@0.5: {metrics['map50']:.4f}")
print(f"  Test mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Test Precision: {metrics['precision']:.4f}")
print(f"  Test Recall: {metrics['recall']:.4f}")
print(f"  Test F1-Score: {metrics['f1']:.4f}")

print("\n" + "=" * 80)
print("SHALLOWER BACKBONE ANALYSIS")
print("=" * 80)
print("\nThis model has a shallower backbone with C2f module repeats")
print("halved at backbone index 6. This creates a lighter model with")
print("faster inference and potentially better generalization on small datasets.")
print("\nCompared with:")
print("  - V1: YOLOv8m Baseline (no modifications)")
print("  - V2: YOLOv8m with Deeper Backbone (added conv layers)")
print("\n✓ Analysis completed")

## Cell 11: Final Summary

In [ ]:
# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED")
print("=" * 80)

print(f"\nExperiment: V3 - YOLOv8 Shallower Backbone")
print(f"Model: YOLOv8{model_config['backbone']} + Shallower")
print(f"Pretrained: {USE_PRETRAINED}")
print(f"Customization: Shallower Backbone (C2f repeats halved)")

print(f"\n" + "-" * 80)
print(f"Training Performance:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")

print(f"\n" + "-" * 80)
print(f"Test Set Performance:")
print(f"  mAP@0.5: {metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"  F1-Score: {metrics['f1']:.4f}")

print(f"\n" + "-" * 80)
print(f"Output Directory: {output_dir}")
print("\n" + "=" * 80)
print("✓ All experiments completed successfully")
print("=" * 80)